In [ ]:
import os
import glob
import pandas as pd

In [ ]:
def dataframe_to_dialogue_txt(df: pd.DataFrame):
    """
    df columns expected:
      - Type: 'P' for Patient, 'T' for Therapist
      - Utterance: text
    Concatenates consecutive same-speaker utterances and returns formatted lines.
    """
    df = df.reset_index(drop=True).copy()
    df["Utterance"] = df["Utterance"].fillna("").astype(str).str.strip()
    df["chunk_id"] = (df["Type"] != df["Type"].shift(1)).cumsum()

    chunks = (
        df.groupby(["chunk_id", "Type"], as_index=False)["Utterance"]
          .apply(lambda s: " ".join([x for x in s if x]))
          .rename(columns={"Utterance": "combined"})
    )

    speaker_map = {"P": "Patient", "T": "Therapist"}
    lines = []
    for _, row in chunks.iterrows():
        label = speaker_map.get(row["Type"], str(row["Type"]))
        text = row["combined"]
        if text:
            lines.append(f"<{label}>: {text}")
    return lines

In [ ]:
# Root of the project — adjust if running from a different working directory
PROJECT_DIR = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
HOPE_DIR    = os.path.join(PROJECT_DIR, "HOPE_WSDM_2022")

# Train is already processed; only run Validation and Test
SPLITS = ["Validation", "Test"]

In [ ]:
for split in SPLITS:
    split_dir = os.path.join(HOPE_DIR, split)
    files = sorted(glob.glob(os.path.join(split_dir, "*.csv")))
    print(f"{split}: {len(files)} CSV files")

    for i, file in enumerate(files):
        df = pd.read_csv(file)
        lines = dataframe_to_dialogue_txt(df)
        out_path = os.path.join(split_dir, f"Transcript_{i+1}.txt")
        with open(out_path, "w", encoding="utf-8") as f:
            f.write("\n\n".join(lines))

    print(f"{split}: wrote {len(files)} transcript .txt files -> {split_dir}")